### Đoạn code hoàn chỉnh để xây dựng bảng DIM_DATE từ các file CSV dữ liệu

In [7]:
import pandas as pd
import glob

def create_dim_date():
    # 1. Định nghĩa các nguồn dữ liệu chứa các cột ngày tháng
    date_sources = {
        '/content/shipments_new.csv': ['ship_date', 'delivery_date'],
        '/content/shipper_new.csv': ['join_date'],
        '/content/review_new.csv': ['review_date'],
        '/content/returns_new.csv': ['return_date'],
        '/content/web_traffic_new.csv': ['date'],
        '/content/customers_new.csv': ['signup_date'],
        '/content/promotion_new.csv': ['start_date', 'end_date'],
        '/content/orders_new.csv': ['order_date'],
        '/content/inventory_new.csv': ['snapshot_date']
    }

    all_dates = set()

    # 2. Quét và thu thập tất cả các ngày duy nhất trong các file
    for file_path, cols in date_sources.items():
        try:
            df = pd.read_csv(file_path, usecols=cols)
            for col in cols:
                # Chuyển đổi về kiểu ngày (date) và loại bỏ giá trị null
                dates = pd.to_datetime(df[col], errors='coerce').dropna().dt.date
                all_dates.update(dates)
        except Exception as e:
            print(f"Lỗi khi đọc file {file_path}: {e}")

    # 3. Khởi tạo DataFrame từ danh sách ngày đã sắp xếp
    sorted_dates = sorted(list(all_dates))
    dim_date = pd.DataFrame({'full_date': sorted_dates})
    dim_date['full_date'] = pd.to_datetime(dim_date['full_date'])

    # 4. Trích xuất các thuộc tính thời gian theo thiết kế StarModel.mmd
    dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
    dim_date['day_of_week'] = dim_date['full_date'].dt.dayofweek + 1  # Thứ 2 (1) -> Chủ nhật (7)
    dim_date['day_name'] = dim_date['full_date'].dt.day_name()
    dim_date['day_of_month'] = dim_date['full_date'].dt.day
    dim_date['month'] = dim_date['full_date'].dt.month
    dim_date['month_name'] = dim_date['full_date'].dt.month_name()
    dim_date['quarter'] = dim_date['full_date'].dt.quarter
    dim_date['year'] = dim_date['full_date'].dt.year
    dim_date['is_weekend'] = dim_date['full_date'].dt.dayofweek.isin([5, 6])

    # Định dạng lại cột full_date sang định dạng chuẩn YYYY-MM-DD
    dim_date['full_date'] = dim_date['full_date'].dt.strftime('%Y-%m-%d')

    # 5. Sắp xếp các cột theo thứ tự thiết kế
    columns_order = [
        'date_key', 'full_date', 'day_of_week', 'day_name', 'day_of_month',
        'month', 'month_name', 'quarter', 'year', 'is_weekend'
    ]
    dim_date = dim_date[columns_order]

    # 6. Lưu kết quả ra file CSV
    output_path = '/content/dim_date_new.csv'
    dim_date.to_csv(output_path, index=False)

    print(f"[Thành công] Đã tạo bảng DIM_DATE với {len(dim_date)} dòng (từ {dim_date['year'].min()} đến {dim_date['year'].max()}).")
    print(f"Lưu file tại: {output_path}")
    return dim_date

# Chạy hàm tạo bảng
dim_date_df = create_dim_date()
display(dim_date_df.head(10))

[Thành công] Đã tạo bảng DIM_DATE với 3985 dòng (từ 2012 đến 2026).
Lưu file tại: /content/dim_date_new.csv


,date_key,full_date,day_of_week,day_name,day_of_month,month,month_name,quarter,year,is_weekend
0,20120117,2012-01-17,2,Tuesday,17,1,January,1,2012,False
1,20120120,2012-01-20,5,Friday,20,1,January,1,2012,False
2,20120202,2012-02-02,4,Thursday,2,2,February,1,2012,False
3,20120205,2012-02-05,7,Sunday,5,2,February,1,2012,True
4,20120207,2012-02-07,2,Tuesday,7,2,February,1,2012,False
5,20120208,2012-02-08,3,Wednesday,8,2,February,1,2012,False
6,20120213,2012-02-13,1,Monday,13,2,February,1,2012,False
7,20120215,2012-02-15,3,Wednesday,15,2,February,1,2012,False
8,20120217,2012-02-17,5,Friday,17,2,February,1,2012,False
9,20120218,2012-02-18,6,Saturday,18,2,February,1,2012,True
